# Holdout Evaluation: Suzuka 2024

This notebook demonstrates an honest test on Suzuka 2024, which was excluded from model training.

**Pipeline**
- Load: `driver_race_features.csv` and the retrained `final_model_logreg.joblib` (trained without Suzuka 2024).
- Holdout selection: filter `season == 2024` and `circuitId == 22` (Suzuka); 
- Encoding: rebuild the training feature space (one-hot dummies) and align the holdout columns to the model’s expected schema.
- Predict: compute top-10 finish probabilities for each driver on Suzuka 2024.
- Compare: view predictions alongside actual `target_top10` to see how the model generalizes to this unseen race.

**Notes**
- Ensure `data/processed/holdout_suzuka_2024.csv` is populated and the model was retrained after excluding these rows. --> done in 03


In [32]:
import pandas as pd
import joblib
from pathlib import Path

FEATURES_PATH = Path("../data/processed/driver_race_features.csv")
HOLDOUT_PATH = Path("../data/processed/holdout_suzuka_2024.csv")
MODEL_PATH = Path("../data/processed/final_model_logreg.joblib")

if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Holdout file missing: {HOLDOUT_PATH}. Re-run notebook 03 after creating the holdout.")

features = pd.read_csv(FEATURES_PATH)
holdout_df = pd.read_csv(HOLDOUT_PATH)
model = joblib.load(MODEL_PATH)

print("Features shape:", features.shape)
print("Holdout shape:", holdout_df.shape)
print("Model:", type(model))

Features shape: (25121, 17)
Holdout shape: (20, 17)
Model: <class 'sklearn.pipeline.Pipeline'>


In [28]:
# Define columns and holdout mask to match training drop
HOLDOUT_SEASON = 2024
HOLDOUT_CIRCUIT = "suzuka"

mask = (features["season"] == HOLDOUT_SEASON) & (features["circuitId"] == HOLDOUT_CIRCUIT)

drop_cols = [
    "target_top10",
    "race_date",
    "driver_name",
    "constructor_name",
    "circuit_name",
    "raceId",
]
cat_cols = ["circuitId", "era", "grid_bucket"]

In [29]:
# Build training feature space (same as training notebook, without holdout)
train_X = features.loc[~mask].drop(columns=drop_cols, errors="ignore")
train_X = pd.get_dummies(train_X, columns=cat_cols, drop_first=True)
train_cols = train_X.columns
print("Train feature space:", train_X.shape)

Train feature space: (25121, 91)


In [30]:
# Prepare holdout features and align columns
X_new = holdout_df.drop(columns=drop_cols, errors="ignore")
X_new = pd.get_dummies(X_new, columns=cat_cols, drop_first=True)

for col in train_cols:
    if col not in X_new:
        X_new[col] = 0
X_new = X_new[train_cols]

print("Holdout matrix:", X_new.shape)

Holdout matrix: (20, 91)


In [31]:
# Predict probabilities and compare with actual target
proba = model.predict_proba(X_new)[:, 1]

preds = holdout_df[["driver_name", "constructor_name", "grid", "target_top10"]].copy()
preds["p_top10"] = proba
preds = preds.sort_values("p_top10", ascending=False).reset_index(drop=True)

preds

,driver_name,constructor_name,grid,target_top10,p_top10
0,Sergio Pérez,Red Bull,2,1,0.863153
1,Max Verstappen,Red Bull,1,1,0.841308
2,Fernando Alonso,Aston Martin,5,1,0.823484
3,Lando Norris,McLaren,3,1,0.800724
4,Carlos Sainz,Ferrari,4,1,0.783793
5,Lewis Hamilton,Mercedes,7,1,0.757579
6,Charles Leclerc,Ferrari,8,1,0.692449
7,Oscar Piastri,McLaren,6,1,0.663345
8,Valtteri Bottas,Sauber,13,0,0.573732
9,George Russell,Mercedes,9,1,0.565731
